# Vista 2M Dense — Google Colab Foundation Training

This notebook runs Vista 2M Dense foundation training on a Google Colab GPU runtime:
1. Verifies CUDA GPU availability (fails closed on CPU runtimes).
2. Optionally mounts Google Drive so checkpoints and `best_model.pt` survive Colab disconnects.
3. Clones/updates the repository and runs the language training verification tests.
4. Launches `tools/train_language_reasoner.py` with `--device cuda --require-cuda`, 2M-token warmup, and 2M-token exam cadence (auto-resuming if `training_state.pt` already exists).
5. Runs `python -m src.language.fx_sentiment_probe` on the resulting checkpoint.

In [ ]:
import platform, sys
import torch

print({'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__, 'cuda': torch.cuda.is_available()})
assert torch.cuda.is_available(), 'CUDA_REQUIRED_SELECT_COLAB_GPU_RUNTIME'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BUNDLE_DIR = '/content/drive/MyDrive/vista_2m_dense_colab'
else:
    BUNDLE_DIR = '/content/vista_2m_dense_colab'
print('BUNDLE_DIR:', BUNDLE_DIR)

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/vistaphantor/fx.git'
BRANCH = 'main'
ROOT = Path('/content/fx')

if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(ROOT)
print('HEAD:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'src/language', 'corpus', 'tools/train_language_reasoner.py'], check=True)
TESTS = [
    'tests/test_language_training_budget.py',
    'tests/test_language_training_pipeline.py',
    'tests/test_language_generation.py',
    'tests/test_language_semantic_checkpointing.py',
    'tests/test_language_fx_sentiment_probe.py',
]
subprocess.run([sys.executable, '-m', 'pytest', *TESTS, '-q'], check=True)

In [ ]:
import subprocess, sys, time
from pathlib import Path

bundle = Path(BUNDLE_DIR)
state_path = bundle / '.training' / 'training_state.pt'

command = [
    sys.executable, 'tools/train_language_reasoner.py',
    '--profile', '2m_dense',
    '--training-stage', 'foundation',
    '--hf-config', 'config/hf_sources.json',
    '--bundle-dir', str(bundle),
    '--device', 'cuda',
    '--require-cuda',
    '--warmup-tokens', '2000000',
    '--exam-every-tokens', '2000000',
    '--session-hours', '10.5',
    '--threads', '4',
]
if state_path.exists():
    command.append('--resume')

print('Launching:', ' '.join(command), flush=True)
started = time.time()
subprocess.run(command, check=True)
print('Training session seconds:', round(time.time() - started, 1))

In [ ]:
import subprocess, sys
from pathlib import Path

bundle = Path(BUNDLE_DIR)
best_ckpt = bundle / '.training' / 'best_model.pt'
assert best_ckpt.exists() or (bundle / 'manifest.json').exists(), 'NO_CHECKPOINT_OR_BUNDLE_FOUND'

if best_ckpt.exists():
    subprocess.run([
        sys.executable, '-m', 'src.language.fx_sentiment_probe',
        '--bundle-dir', str(bundle),
        '--device', 'cuda',
        '--require-cuda',
    ], check=True)